<a href="https://colab.research.google.com/github/NicholasVunZhunMin/PL/blob/main/%E2%80%9CHW1_%E6%97%A5%E5%B8%B8%E6%94%AF%E5%87%BA%E9%80%9F%E7%AE%97%E8%88%87%E5%88%86%E6%94%A4_Gradio_Part2%E2%80%9D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#日常支出速算與分攤（作業一）
- 目標：從 Sheet 讀「消費紀錄」→ 計總額/分類小計/AA 分攤 → 寫回 Sheet Summary 分頁。
- AI 點子（可選）：請模型總結本週花錢習慣與建議（例如「外食過多」）。
- Sheet 欄位：date, category, item, amount, payer

GoogleSheet: https://docs.google.com/spreadsheets/d/1jR3qRQr2ZvWYKNuv8wen_-eTZWdc5a-LLvH7iymn2zw/edit?usp=sharing

In [ ]:
import gradio as gr
import pandas as pd
import datetime
import gspread
import matplotlib.pyplot as plt
from google.colab import auth
from google.auth import default

# --- 設定區 ---
SHEET_URL = "https://docs.google.com/spreadsheets/d/1jR3qRQr2ZvWYKNuv8wen_-eTZWdc5a-LLvH7iymn2zw/edit?usp=sharing"
WORKSHEET_NAME = "工作表1"
SUMMARY_SHEET_NAME = "Summary"
REQUIRED_COLUMNS = ["日期", "時間", "分類", "品項", "金額", "付款人"]

# --- 初始化與授權 ---
_auth_done = False
_gc = None
_ws = None

def _ensure_auth_and_ws():
    global _auth_done, _gc, _ws
    if not _auth_done:
        auth.authenticate_user()
        creds, _ = default()
        _gc = gspread.authorize(creds)
        _auth_done = True
    if _ws is None:
        gs = _gc.open_by_url(SHEET_URL)
        try:
            _ws = gs.worksheet(WORKSHEET_NAME)
        except:
            _ws = gs.add_worksheet(title=WORKSHEET_NAME, rows="1000", cols="20")
        _ensure_headers(_ws)
    return _ws

def _ensure_headers(ws):
    rows = ws.get_all_values()
    if not rows or rows[0][:len(REQUIRED_COLUMNS)] != REQUIRED_COLUMNS:
        ws.update('1:1', [REQUIRED_COLUMNS])

# --- 核心邏輯：數據處理與 AI ---

def _read_df():
    ws = _ensure_auth_and_ws()
    values = ws.get_all_values()
    if len(values) <= 1:
        return pd.DataFrame(columns=REQUIRED_COLUMNS)
    df = pd.DataFrame(values[1:], columns=values[0])
    df["金額"] = pd.to_numeric(df["金額"], errors="coerce").fillna(0.0)
    return df

def get_ai_advice(df):
    if df.empty or df["金額"].sum() == 0: return "目前還沒有消費數據，快去記帳吧！☕"
    total = df["金額"].sum()
    cat_sums = df.groupby("分類")["金額"].sum()
    top_cat = cat_sums.idxmax()
    percent = (cat_sums.max() / total) * 100

    advice = f"### 💡 AI 理財小語\n"
    advice += f"本期累計支出 **${total:,.0f}**。其中 **【{top_cat}】** 佔了 {percent:.1}％，是最大的開銷項目。"
    if percent > 50: advice += "\n⚠️ 某個項目比例過高，建議檢查是否有非必要支出喔！"
    elif total > 10000: advice += "\n💸 最近花錢有點兇，要稍微節制一下囉！"
    else: advice += "\n✨ 支出分配看起來還算平衡，繼續保持！"
    return advice

def create_pie_chart(df):
    if df.empty or df["金額"].sum() == 0: return None
    plt.figure(figsize=(6, 4))
    # 解決 Matplotlib 中文顯示問題 (Colab 環境)
    plt.rcParams['font.sans-serif'] = ['Liberation Sans']
    cat_data = df.groupby("分類")["金額"].sum()
    plt.pie(cat_data, labels=cat_data.index, autopct='%1.1f%%', startangle=140, colors=['#ff9999','#66b3ff','#99ff99','#ffcc99'])
    plt.title("消費分類佔比")
    return plt

def update_summary_to_sheet(df, by_cat, settle):
    """將結果寫回 Google Sheet 的 Summary 分頁"""
    try:
        gs = _gc.open_by_url(SHEET_URL)
        try:
            s_ws = gs.worksheet(SUMMARY_SHEET_NAME)
        except:
            s_ws = gs.add_worksheet(title=SUMMARY_SHEET_NAME, rows="100", cols="10")

        s_ws.clear()
        # 寫入分類統計
        s_ws.update('A1', [["分類統計報告"]])
        s_ws.update('A2', [by_cat.columns.tolist()] + by_cat.values.tolist())
        # 寫入 AA 結算 (放在分類統計旁邊或下方)
        s_ws.update('E1', [["AA 分攤結算"]])
        s_ws.update('E2', [settle.columns.tolist()] + settle.values.tolist())
        return "✅ 已自動同步至 Sheet Summary 分頁"
    except Exception as e:
        return f"⚠️ Sheet 同步失敗: {e}"

# --- Gradio 串接函數 ---

def add_and_refresh(date, time, cat, item, amt, payer):
    ws = _ensure_auth_and_ws()
    ws.append_row([date, time, cat, item, float(amt), payer], value_input_option="USER_ENTERED")

    # 重新讀取並計算
    df = _read_df()
    by_cat = df.groupby("分類", as_index=False)["金額"].sum().sort_values("金額", ascending=False)

    # AA 分攤計算
    unique_payers = [p for p in df["付款人"].unique() if p]
    n = max(len(unique_payers), 1)
    aa_share = df["金額"].sum() / n
    paid_by = df.groupby("付款人", as_index=False)["金額"].sum()
    settle = pd.DataFrame({"付款人": unique_payers})
    settle = settle.merge(paid_by, on="付款人", how="left").fillna(0)
    settle.columns = ["付款人", "實付"]
    settle["應付(AA)"] = aa_share
    settle["差額"] = settle["實付"] - settle["應付(AA)"]

    sheet_msg = update_summary_to_sheet(df, by_cat, settle)
    advice = get_ai_advice(df)
    chart = create_pie_chart(df)

    return f"✅ 已新增！{sheet_msg}", df["金額"].sum(), by_cat, settle, advice, chart

# --- UI 介面 ---
with gr.Blocks(theme=gr.themes.Soft(), title="智能家計簿") as demo:
    gr.Markdown("# 🧾 智能家計簿 & 自動分攤系統")

    with gr.Tab("➕ 快速記帳"):
        with gr.Row():
            with gr.Column():
                date_in = gr.Textbox(label="📅 日期", value=datetime.date.today().strftime("%Y-%m-%d"))
                time_in = gr.Textbox(label="⏰ 時間", placeholder="HH:MM (選填)")
                cat_in = gr.Dropdown(label="📂 分類", choices=["外食", "交通", "購物", "娛樂", "房租", "其他"], allow_custom_value=True)
                item_in = gr.Textbox(label="🛍️ 品項")
                amt_in = gr.Number(label="💰 金額")
                payer_in = gr.Textbox(label="👤 付款人")
                add_btn = gr.Button("🚀 點我記帳", variant="primary")

            with gr.Column():
                status_msg = gr.Markdown("等待輸入...")
                ai_box = gr.Markdown("### 💡 AI 建議將在此顯示")
                plot_out = gr.Plot(label="消費分佈圖")

        with gr.Row():
            cat_df = gr.Dataframe(label="📊 分類小計")
            settle_df = gr.Dataframe(label="🤝 AA 分攤結算")

        add_btn.click(
            add_and_refresh,
            [date_in, time_in, cat_in, item_in, amt_in, payer_in],
            [status_msg, gr.Number(visible=False), cat_df, settle_df, ai_box, plot_out]
        )

    with gr.Tab("📒 歷史明細"):
        refresh_btn = gr.Button("🔍 刷新所有資料")
        full_df = gr.Dataframe()
        refresh_btn.click(_read_df, None, full_df)

demo.launch(share=True, debug=True)

/tmp/ipykernel_1011/3822623654.py:124: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="智能家計簿") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1837217753f7c42ae8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1IJ4Pt5SDfWjgyuEW51LYuHmv4IVnAbDipr1b3100u_Q/edit?usp=sharing"
WORKSHEET_NAME = "工作表1"

REQUIRED_COLUMNS = ["日期", "時間", "分類", "品項", "金額", "付款人"]

_auth_done = False
_gc = None
_ws = None